<h1 style=\"text-align: center; font-size: 50px;\"> Register Model </h1>

# Notebook Overview

- Start Execution
- Define User Constants
- Install and Import Libraries
- Configure Settings
- Verify Assets
- Load and Validate Data
- Define MLflow Class
- Log the Model to MLflow
- Fetch the Latest Model Version from MLflow
- Load the Model and Run Inference
- Display Evaluation Results
- Save Evaluation Results

# Start Execution

In [1]:
# -----------------------------
# Standard library imports
# -----------------------------
import json                 # JSON parsing and serialization
import os                   # Operating system utilities (paths, env vars, etc.)
import sys                  # Python runtime environment manipulation
import time                 # Time-related utilities
from datetime import datetime  # Date and time handling
from pathlib import Path     # Object-oriented filesystem paths

# -----------------------------
# Local imports
# -----------------------------
# Extend sys.path to allow importing from parent directory
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.utils import logger  # Project-specific logging utility

In [2]:
start_time = time.time()  
logger.info("Notebook execution started.")

# Define User Constants

In [3]:
# File configuration
INPUT_FILE_NAME: str = "2025 ISEF Project Abstracts.csv"
INPUT_DIR: Path = Path("../data/inputs")
OUTPUT_DIR: Path = Path("../data/outputs")

# Ensure directories exist
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH: Path = INPUT_DIR / INPUT_FILE_NAME
TIMESTAMP: str = datetime.now().strftime('%Y-%m-%d %H-%M-%S')
OUTPUT_FILE_NAME: str = f"Evaluated - {INPUT_FILE_NAME} - {TIMESTAMP}.csv"
OUTPUT_PATH: Path = OUTPUT_DIR / OUTPUT_FILE_NAME

# Evaluation configuration
KEY_COLUMN: str = "title"
EVAL_COLUMN: str = "abstract"
CRITERIA: dict[str, int] = json.loads(
        json.dumps({
            "Originality": 3,
            "ScientificRigor": 4,
            "Clarity": 2,
            "Relevance": 1,
            "Feasibility": 3,
            "Brevity": 2,
        }),
)

# Install and Import Libraries

In [4]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 10.7 ms, sys: 8.81 ms, total: 19.5 ms
Wall time: 816 ms


In [5]:
import os
import json
import logging
import multiprocessing
import sys
from pathlib import Path
import warnings
import re
from typing import Any, Dict, List

import pandas as pd
import numpy as np
import mlflow
import mlflow.pyfunc
from mlflow.models import ModelSignature
from mlflow.types import Schema, ColSpec, DataType, ParamSpec, ParamSchema, TensorSpec
from mlflow.tracking import MlflowClient
from llama_cpp import Llama
from tqdm.auto import tqdm
import os
import re
import sys
import warnings
import multiprocessing
from typing import Any, Dict, List

import pandas as pd
from tqdm.auto import tqdm
from llama_cpp import Llama

# Add src directory to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from src.utils import load_config, configure_proxy

# Configure Settings

In [6]:
warnings.filterwarnings("ignore")

In [7]:
EXPERIMENT_NAME = "EvaluationExperiment"
RUN_NAME = "EvaluationRun"
MODEL_NAME = "EvaluationModel"

LLAMA_MODEL_PATH = "/home/jovyan/datafabric/meta-llama3.1-8b-Q8/Meta-Llama-3.1-8B-Instruct-Q8_0.gguf"

# Load configuration
CONFIG_PATH = "../configs/config.yaml"
config = load_config(CONFIG_PATH)

# Configure proxy if specified
configure_proxy(config)

print("✅ Configuration loaded successfully")

✅ Configuration loaded successfully


# Verify Assets

In [8]:
def log_asset_status(asset_path: str, asset_name: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured.")
    else:
        logger.info(f"{asset_name} is not properly configured. Please ensure the required asset is correctly configured in your AI Studio project according to the README file.")

In [9]:
log_asset_status(
    asset_path=INPUT_PATH,
    asset_name="Input Data",
)

log_asset_status(
    asset_path=LLAMA_MODEL_PATH,
    asset_name="LLaMA Local model",
)

# Load and Validate Data

In [10]:
df = pd.read_csv(INPUT_PATH)

df.head(10)

,title,category,year,schools,abstract,country,State,Province,awards
0,Dynamic Response of a Human Neck Replica to Ax...,Energy: Physical,2014,set(),Purpose: A human neck replica was made to simu...,United States of America,MN,NaN,['nan']
1,The Effect of Nutrient Solution Concentration ...,Physics and Astronomy,2014,set(),Studies comparing the mineral nutrition of hyd...,United States of America,UT,NaN,['nan']
2,Do Air Root Pruning Pots Accelerate Success in...,Physics and Astronomy,2014,set(),The purpose of my project was to determine whi...,United States of America,LA,NaN,['nan']
3,Insect-repelling Plants & New Organic Pesticide,Environmental Engineering,2014,set(),Organochlorine pesticides in agriculture are n...,United States of America,TX,NaN,['nan']
4,How Do Different Factors Affect the Accuracy o...,Earth and Environmental Sciences,2014,set(),The purpose of this experiment is to determine...,United States of America,MN,NaN,['nan']
5,Dye Sensitized Solar Cells: New Structures and...,Engineering Mechanics,2014,set(),Although fossil fuels have the capacity to pow...,United States of America,TX,NaN,['Fourth Award of $500']
6,A Novel Method for Determination of Camera Pos...,Embedded Systems,2014,set(),The method proposed here solves for the pose o...,United States of America,MO,NaN,['nan']
7,Observational Detection of Solar g-mode Oscill...,Microbiology,2014,set(),NaN,United States of America,HI,NaN,"['Third Award of $1,000']"
8,Synthesis of Periodic Mesoporous Organosilicas...,Plant Sciences,2014,set(),NaN,United States of America,TX,NaN,"['Second Award of $2,000']"
9,A Novel Approach to Solar Desalination Using N...,Plant Sciences,2014,set(),The purpose of the project was to determine if...,United States of America,FL,NaN,['nan']


In [11]:
# Validate required columns
missing_columns: list[str] = [
    col for col in [KEY_COLUMN, EVAL_COLUMN] if col not in df.columns
]
if missing_columns:
    raise KeyError(f"Missing required column(s): {', '.join(missing_columns)}")

# Ensure key column is of string type
df[KEY_COLUMN] = df[KEY_COLUMN].astype(str)

# Define MLflow Class

In [12]:
class EvaluatorModel(mlflow.pyfunc.PythonModel):
    """
    A PythonModel using a local LLaMA model to evaluate texts by multiple criteria.
    """
    def load_context(self, context):
        """Load LLaMA model from artifacts with optimized configuration."""
        model_path = context.artifacts["llama_model_path"]
        self.llm = Llama(
            model_path=model_path,
            n_gpu_layers=-1,
            n_batch=128,
            n_ctx=8192,
            max_tokens=512,
            f16_kv=True,
            use_mmap=False,
            low_vram=True,
            rope_scaling=None,
            temperature=0.0,
            repeat_penalty=1.0,
            streaming=False,
            stop=None,
            seed=42,
            num_threads=multiprocessing.cpu_count(),
            verbose=False,
        )

    # ─── Helper Functions ───────────────────────────────────────────────────────

    def scale_score(self, raw_score: int, max_target: int) -> int:
        """
        Scales a score from a 1–10 range to the given max_target range.
        Clamps the result between 0 and max_target.
        """
        scaled: int = round((raw_score / 10) * max_target)
        return min(max(scaled, 0), max_target)
    
    
    def extract_single_score(self, output: str) -> int:
        """
        Extracts a single integer score (1–10) from the LLM output.
        Returns -1 if no valid score is found.
        """
        match = re.search(r"\b(10|[1-9])\b", output)
        return int(match.group(1)) if match else -1
    
    
    def evaluate_criterion(self, text: str, criterion: str) -> int:
        """
        Prompts the LLM to score the text based on a specific criterion.
        Returns the extracted integer score from the LLM's response.
        """
        if not isinstance(text, str):
            return -1
    
        prompt: str = (
            f"You are an expert evaluator. Rate the abstract below based solely on the criterion: '{criterion}'.\n"
            "Provide a single integer from 1 to 10 (inclusive).\n"
            "Output only the number — no words, labels, punctuation, or explanations.\n\n"
            f"Abstract:\n{text.strip()}\n\n"
            "Score:"
        )
    
    
        response: str = self.llm(prompt)["choices"][0]["text"]
        return self.extract_single_score(response)
    
    
    def evaluate_row(self, text: str, criteria: Dict) -> Dict[str, int]:
        """
        Evaluates a single text against all rubric criteria.
        Returns a dictionary of scaled scores.
        """
        return {
            criterion: self.scale_score(
                self.evaluate_criterion(text, criterion),
                criteria[criterion]
            )
            for criterion in criteria
        }

    def predict(self, context, model_input: pd.DataFrame, params: dict) -> pd.DataFrame:
        """Evaluate texts using LLaMA model and return scores with total."""
        # Extract parameters
        key_col = params.get("key_column", KEY_COLUMN)
        eval_col = params.get("eval_column", EVAL_COLUMN)
        criteria = params.get("criteria", CRITERIA)
        if isinstance(criteria, str):
            criteria = json.loads(criteria)

        # Validate input
        for col in (key_col, eval_col):
            if col not in model_input.columns:
                raise KeyError(f"Input DataFrame missing column '{col}'")

        df = model_input.copy()
        df[key_col] = df[key_col].astype(str)

        evaluation_results: list[dict[str, Any]] = []

        for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating rows"):
            evaluated_row: dict[str, Any] = self.evaluate_row(row[EVAL_COLUMN], criteria)
            evaluated_row[KEY_COLUMN] = row[KEY_COLUMN]
            evaluation_results.append(evaluated_row)
        
        # Convert results to a DataFrame
        evaluation_df: pd.DataFrame = pd.DataFrame(evaluation_results)

        # Merge original data with evaluation results on the key column
        final_df: pd.DataFrame = df.merge(evaluation_df, on=KEY_COLUMN)
        
        # Compute total score by summing across all criteria
        final_df["TotalScore"] = final_df[list(CRITERIA.keys())].sum(axis=1)
        
        # Sort the DataFrame by total score in descending order
        final_df.sort_values(by="TotalScore", ascending=False, inplace=True)

        return final_df

    @classmethod
    def log_model(
        cls, 
        model_name: str, 
        llama_model_path: str, 
        config_path: str,
        experiment_name: str = EXPERIMENT_NAME
    ):
        """
        Logs and registers this model in MLflow.
        """
        # Define artifacts
        artifacts = {
            "llama_model_path": llama_model_path,
            "config_path": config_path,
            "demo": "../demo",
            }

        params_schema = ParamSchema([
            ParamSpec("key_column",  DataType.string,  'title'),
            ParamSpec("eval_column", DataType.string,  'abstract'),
            ParamSpec("criteria",    DataType.string,  '["Originality","Clarity","Relevance","Feasibility","Feasibility"]'),
        ])
        
        signature = ModelSignature(inputs=None, outputs=None, params=params_schema)

        mlflow.pyfunc.log_model(
            artifact_path=model_name,
            python_model=cls(),
            artifacts=artifacts,
            signature=signature,
            registered_model_name=model_name,
            pip_requirements='../requirements.txt'
        )
        logger.info(f"Model '{model_name}' logged and registered.")

# Log the Model to MLflow

In [13]:
%%time

mlflow.set_tracking_uri('/phoenix/mlflow')
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run(run_name=RUN_NAME) as run:
    run_id = run.info.run_id
    logger.info("Run ID: %s", run_id)

    EvaluatorModel.log_model(
        model_name=MODEL_NAME,
        llama_model_path=LLAMA_MODEL_PATH,
        config_path=CONFIG_PATH,
    )

    # mlflow.register_model(
    #     model_uri=f"runs:/{run_id}/{MODEL_NAME}",
    #     name=MODEL_NAME
    # )
    logger.info("Registered model: %s", MODEL_NAME)

2025/10/30 21:23:58 INFO mlflow.tracking.fluent: Experiment with name 'EvaluationExperiment' does not exist. Creating a new experiment.


Successfully registered model 'EvaluationModel'.
Created version '1' of model 'EvaluationModel'.


CPU times: user 852 ms, sys: 27.6 s, total: 28.4 s
Wall time: 4min 47s


# Fetch the Latest Model Version from MLflow

In [14]:
# Load latest model
client = MlflowClient()
latest_version = client.get_latest_versions(MODEL_NAME, stages=["None"])[0].version
logger.info(f"Latest model version: {latest_version}")

# Load the Model and Run Inference

In [15]:
%%time

model_uri = f"models:/{MODEL_NAME}/{latest_version}"
model = mlflow.pyfunc.load_model(model_uri)

CPU times: user 1.88 s, sys: 2.71 s, total: 4.59 s
Wall time: 1min 22s


In [16]:
%%time

# Sample input
sample = df
params = {
    "key_column": KEY_COLUMN,
    "eval_column": EVAL_COLUMN,
    "criteria": json.dumps(CRITERIA),
}
preds = model.predict(sample, params=params)

Evaluating rows:   0%|          | 0/10 [00:00<?, ?it/s]

CPU times: user 26.9 s, sys: 0 ns, total: 26.9 s
Wall time: 27.1 s


# Display Evaluation Results

In [17]:
# Preview the top 10 evaluated entries
preds.head(10)

,title,category,year,schools,abstract,country,State,Province,awards,Originality,ScientificRigor,Clarity,Relevance,Feasibility,Brevity,TotalScore
0,Dynamic Response of a Human Neck Replica to Ax...,Energy: Physical,2014,set(),Purpose: A human neck replica was made to simu...,United States of America,MN,NaN,['nan'],1,4,2,1,3,0,11
1,The Effect of Nutrient Solution Concentration ...,Physics and Astronomy,2014,set(),Studies comparing the mineral nutrition of hyd...,United States of America,UT,NaN,['nan'],1,4,2,1,3,0,11
3,Insect-repelling Plants & New Organic Pesticide,Environmental Engineering,2014,set(),Organochlorine pesticides in agriculture are n...,United States of America,TX,NaN,['nan'],1,2,1,1,2,2,9
5,Dye Sensitized Solar Cells: New Structures and...,Engineering Mechanics,2014,set(),Although fossil fuels have the capacity to pow...,United States of America,TX,NaN,['Fourth Award of $500'],1,3,1,1,2,1,9
6,A Novel Method for Determination of Camera Pos...,Embedded Systems,2014,set(),The method proposed here solves for the pose o...,United States of America,MO,NaN,['nan'],1,2,2,1,2,1,9
9,A Novel Approach to Solar Desalination Using N...,Plant Sciences,2014,set(),The purpose of the project was to determine if...,United States of America,FL,NaN,['nan'],1,2,1,1,3,1,9
4,How Do Different Factors Affect the Accuracy o...,Earth and Environmental Sciences,2014,set(),The purpose of this experiment is to determine...,United States of America,MN,NaN,['nan'],1,1,1,1,3,1,8
2,Do Air Root Pruning Pots Accelerate Success in...,Physics and Astronomy,2014,set(),The purpose of my project was to determine whi...,United States of America,LA,NaN,['nan'],1,2,1,0,2,1,7
7,Observational Detection of Solar g-mode Oscill...,Microbiology,2014,set(),NaN,United States of America,HI,NaN,"['Third Award of $1,000']",0,0,0,0,0,0,0
8,Synthesis of Periodic Mesoporous Organosilicas...,Plant Sciences,2014,set(),NaN,United States of America,TX,NaN,"['Second Award of $2,000']",0,0,0,0,0,0,0


# Save Evaluation Results

In [18]:
preds.to_csv(OUTPUT_PATH, index=False)
logger.info(f"✅ Evaluation results successfully saved to: {OUTPUT_PATH}")

In [19]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).